# LeetCode 985 — Sum of Even Numbers After Queries bằng Python

Link đề: https://leetcode.com/problems/sum-of-even-numbers-after-queries/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def sumEvenAfterQueries(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        ...
```

Ghi chú: với mỗi truy vấn `[val, index]` ta **cộng `val` vào `nums[index]`** rồi báo cáo **tổng các số chẵn** hiện có trong `nums`. Quét lại cả mảng mỗi truy vấn sẽ tốn `O(n * q)`; thay vào đó ta **duy trì tổng các số chẵn** và chỉ cập nhật phần bị ảnh hưởng bởi **một** phần tử vừa đổi: trừ giá trị cũ nếu nó chẵn, cộng giá trị mới nếu nó chẵn. Bài này xoay quanh một ý nhỏ nhưng hay sai: **cập nhật tăng dần (incremental)** thay vì tính lại từ đầu, và **tính chẵn/lẻ của số âm**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cập nhật tại chỗ rồi báo cáo một thống kê của toàn mảng sau **mỗi** truy vấn.
2. Nắm các công cụ Python cần thiết: toán tử `%` và `&` với số âm (`-3 % 2 == 1`, `-3 & 1 == 1`), `sum(... for ... if ...)`, vòng lặp có cập nhật trạng thái, và mảng hiệu (difference array) cho cách ngoại tuyến.
3. Hiểu nguyên lý **duy trì bất biến**: giữ biến `s` luôn bằng tổng các số chẵn của mảng hiện tại; mỗi truy vấn chỉ đổi **một** phần tử nên chỉ cần "gỡ" đóng góp cũ và "cộng" đóng góp mới của phần tử đó.
4. Đi qua 4 cách: mô phỏng và quét lại cả mảng (brute force), cập nhật tăng dần bằng trừ-cộng (cách nộp chính), phân tích 4 trường hợp chẵn/lẻ của `(cũ, val)` và cách **ngoại tuyến** dùng mảng hiệu theo từng chỉ số.
5. Tránh các lỗi kinh điển: quên **trừ** giá trị cũ, chỉ xét tính chẵn của `val` thay vì của giá trị mới, quên cập nhật `nums[index]`, quên tính tổng ban đầu, quên cộng giá trị mới, và kiểm tra lẻ bằng `x % 2 == 1` trong các ngôn ngữ cho số dư âm.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và mảng `queries` với `queries[i] = [val_i, index_i]`.

Với mỗi truy vấn `i`: trước hết thực hiện `nums[index_i] = nums[index_i] + val_i`, rồi **in ra tổng các giá trị chẵn** của `nums`.

Trả về mảng `answer` với `answer[i]` là đáp án của truy vấn thứ `i`.

**Ví dụ 1:** `nums = [1,2,3,4]`, `queries = [[1,0],[-3,1],[-4,0],[2,3]]` -> `[8,6,2,4]`.

- Ban đầu `[1,2,3,4]`. Cộng `1` vào `nums[0]`: `[2,2,3,4]`, tổng chẵn `2 + 2 + 4 = 8`.
- Cộng `-3` vào `nums[1]`: `[2,-1,3,4]`, tổng chẵn `2 + 4 = 6`.
- Cộng `-4` vào `nums[0]`: `[-2,-1,3,4]`, tổng chẵn `-2 + 4 = 2`.
- Cộng `2` vào `nums[3]`: `[-2,-1,3,6]`, tổng chẵn `-2 + 6 = 4`.

**Ví dụ 2:** `nums = [1]`, `queries = [[4,0]]` -> `[0]`.

**Ràng buộc:**

- `1 <= nums.length <= 10^4`
- `-10^4 <= nums[i] <= 10^4`
- `1 <= queries.length <= 10^4`
- `-10^4 <= val_i <= 10^4`
- `0 <= index_i < nums.length`

In [1]:
from typing import List, Tuple
import math
import random, time

examples = [
    ([1, 2, 3, 4], [[1, 0], [-3, 1], [-4, 0], [2, 3]], [8, 6, 2, 4]),
    ([1], [[4, 0]], [0]),
]
for nums, queries, expected in examples:
    print(f"nums={nums} queries={queries} -> kỳ vọng {expected}")

nums=[1, 2, 3, 4] queries=[[1, 0], [-3, 1], [-4, 0], [2, 3]] -> kỳ vọng [8, 6, 2, 4]
nums=[1] queries=[[4, 0]] -> kỳ vọng [0]


## 3. Kiến thức nền

### 3.1. Chẵn/lẻ với số âm trong Python

Trong Python, `x % 2` luôn cho kết quả **không âm** khi chia cho `2` (dấu theo số chia), nên `-3 % 2 == 1` và `-4 % 2 == 0`; `x % 2 == 0` kiểm tra chẵn đúng cho mọi số nguyên (kể cả `0` và số âm). Phép `x & 1` cho cùng kết quả. Lưu ý: ở C/C++/Java, `-3 % 2 == -1` (dấu theo số bị chia) nên kiểm tra lẻ bằng `x % 2 == 1` **sai** với số âm; luôn kiểm tra lẻ bằng `x % 2 != 0`.

In [2]:
for x in (-4, -3, 0, 3, 4):
    c_rem = int(math.fmod(x, 2))               # mô phỏng phép % kiểu C/Java (dấu theo số bị chia)
    print(f"x={x:2}: x % 2 = {x % 2}, x & 1 = {x & 1}, chẵn? {x % 2 == 0} | số dư kiểu C = {c_rem:2} (lẻ theo '== 1'? {c_rem == 1})")

x=-4: x % 2 = 0, x & 1 = 0, chẵn? True | số dư kiểu C =  0 (lẻ theo '== 1'? False)
x=-3: x % 2 = 1, x & 1 = 1, chẵn? False | số dư kiểu C = -1 (lẻ theo '== 1'? False)
x= 0: x % 2 = 0, x & 1 = 0, chẵn? True | số dư kiểu C =  0 (lẻ theo '== 1'? False)
x= 3: x % 2 = 1, x & 1 = 1, chẵn? False | số dư kiểu C =  1 (lẻ theo '== 1'? True)
x= 4: x % 2 = 0, x & 1 = 0, chẵn? True | số dư kiểu C =  0 (lẻ theo '== 1'? False)


### 3.2. Tổng các số chẵn

`sum(x for x in nums if x % 2 == 0)` là tổng các số chẵn. Số `0` là chẵn (đóng góp `0`); số chẵn âm đóng góp giá trị **âm** nên tổng có thể âm.

In [3]:
nums = [-2, -1, 3, 6]
print("tổng chẵn của", nums, "=", sum(x for x in nums if x % 2 == 0))
print("tổng chẵn của [-2, -4, 3] =", sum(x for x in [-2, -4, 3] if x % 2 == 0), "(âm vẫn hợp lệ)")

tổng chẵn của [-2, -1, 3, 6] = 4
tổng chẵn của [-2, -4, 3] = -6 (âm vẫn hợp lệ)


### 3.3. Cập nhật tăng dần (incremental)

Khi một thống kê chỉ phụ thuộc phần tử theo kiểu **cộng** (tổng của các phần tử thỏa điều kiện), ta không cần tính lại từ đầu: khi **một** phần tử `x` đổi thành `x'`, thống kê đổi đúng bằng `đóng_góp(x') - đóng_góp(x)`. Ở đây `đóng_góp(x) = x` nếu `x` chẵn, `0` nếu `x` lẻ.

In [4]:
def contrib(x: int) -> int:
    return x if x % 2 == 0 else 0

s = 2 + 2 + 4 + 0                                 # tổng chẵn của [2, 2, 3, 4] = 8
old, new = 2, -1                                  # nums[1]: 2 -> -1 (cộng -3)
s += contrib(new) - contrib(old)
print("sau khi đổi nums[1] từ 2 thành -1: tổng chẵn =", s, "(đúng: 2 + 4 = 6)")
assert s == 6

sau khi đổi nums[1] từ 2 thành -1: tổng chẵn = 6 (đúng: 2 + 4 = 6)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `nums = [1,2,3,4]`; tổng chẵn ban đầu `s = 2 + 4 = 6`.

- Truy vấn `[1, 0]`: `nums[0]` từ `1` (lẻ, đóng góp `0`) thành `2` (chẵn, đóng góp `2`): `s = 6 - 0 + 2 = 8`.
- Truy vấn `[-3, 1]`: `nums[1]` từ `2` (chẵn, `2`) thành `-1` (lẻ, `0`): `s = 8 - 2 + 0 = 6`.
- Truy vấn `[-4, 0]`: `nums[0]` từ `2` (chẵn) thành `-2` (chẵn): `s = 6 - 2 + (-2) = 2`.
- Truy vấn `[2, 3]`: `nums[3]` từ `4` (chẵn) thành `6` (chẵn): `s = 2 - 4 + 6 = 4`.

Kết quả `[8, 6, 2, 4]`. Mỗi truy vấn chỉ tốn `O(1)`.

## 5. Cách 1 — Brute Force: cập nhật rồi quét lại cả mảng

**Ý tưởng.** Với mỗi truy vấn: cập nhật `nums[index] += val`, rồi tính lại `sum(x for x in nums if x % 2 == 0)` bằng cách quét **toàn bộ** mảng. Bám sát đúng đề, nên cũng là **oracle độc lập**.

In [5]:
def sum_even_brute(nums: List[int], queries: List[List[int]]) -> List[int]:
    nums = list(nums)                                   # không sửa mảng đầu vào
    answer = []
    for val, idx in queries:
        nums[idx] += val
        answer.append(sum(x for x in nums if x % 2 == 0))
    return answer

for nums, queries, expected in examples:
    got = sum_even_brute(nums, queries)
    print(f"nums={nums}: brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

nums=[1, 2, 3, 4]: brute = [8, 6, 2, 4] OK
nums=[1]: brute = [0] OK


**Độ phức tạp và nút thắt.** `O(n * q)`: với `n = q = 10^4` là `10^8` phép toán: quá chậm trong Python. Đo thử ở cỡ nhỏ rồi ngoại suy:

In [6]:
rng = random.Random(985)
for n in (500, 1000, 2000):
    nums = [rng.randint(-10 ** 4, 10 ** 4) for _ in range(n)]
    queries = [[rng.randint(-10 ** 4, 10 ** 4), rng.randrange(n)] for _ in range(n)]
    t0 = time.perf_counter()
    sum_even_brute(nums, queries)
    dt = time.perf_counter() - t0
    print(f"n=q={n:5}: {dt * 1000:8.1f} ms -> ngoại suy n=q=10^4: ~{dt * (10 ** 4 / n) ** 2:7.1f} s")

n=q=  500:     12.1 ms -> ngoại suy n=q=10^4: ~    4.8 s
n=q= 1000:     47.9 ms -> ngoại suy n=q=10^4: ~    4.8 s


n=q= 2000:    196.2 ms -> ngoại suy n=q=10^4: ~    4.9 s


Nút thắt: mỗi truy vấn chỉ đổi **một** phần tử nhưng ta quét lại **cả mảng**. Ta chỉ cần cập nhật phần bị ảnh hưởng.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ tổng chẵn `s` làm bất biến, mỗi truy vấn cập nhật `O(1)`.**

1. Tính `s = sum(x for x in nums if x % 2 == 0)` một lần ở đầu.
2. Với mỗi truy vấn `[val, idx]`:
   - nếu `nums[idx]` (giá trị **cũ**) chẵn thì `s -= nums[idx]` (gỡ đóng góp cũ);
   - `nums[idx] += val`;
   - nếu `nums[idx]` (giá trị **mới**) chẵn thì `s += nums[idx]` (thêm đóng góp mới);
   - ghi `s` vào đáp án.

Tổng thời gian `O(n + q)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Trừ giá trị cũ nếu chẵn, cộng giá trị mới nếu chẵn (nộp) |
| 3 | Phân tích 4 trường hợp chẵn/lẻ của `(giá trị cũ, val)` để cập nhật thẳng |
| 4 | Ngoại tuyến: với mỗi chỉ số, đóng góp là hằng số trên từng đoạn thời gian; cộng vào **mảng hiệu** |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Trừ giá trị cũ, cộng giá trị mới (cách nộp chính)

**Giải thích từng đoạn.**

- `s` là tổng chẵn của `nums` hiện tại; tính một lần ở đầu.
- Nếu giá trị cũ chẵn: `s -= nums[idx]` (nó sắp bị thay).
- Cập nhật `nums[idx] += val`.
- Nếu giá trị mới chẵn: `s += nums[idx]`.
- `answer.append(s)`.

In [7]:
def sum_even_incremental(nums: List[int], queries: List[List[int]]) -> List[int]:
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    answer = []
    for val, idx in queries:
        if nums[idx] % 2 == 0:
            s -= nums[idx]
        nums[idx] += val
        if nums[idx] % 2 == 0:
            s += nums[idx]
        answer.append(s)
    return answer

for nums, queries, expected in examples:
    got = sum_even_incremental(nums, queries)
    print(f"nums={nums}: tăng dần = {got}", "OK" if got == expected else "SAI")
    assert got == expected

nums=[1, 2, 3, 4]: tăng dần = [8, 6, 2, 4] OK
nums=[1]: tăng dần = [0] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def sumEvenAfterQueries(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        s = sum(x for x in nums if x % 2 == 0)          # tổng chẵn ban đầu
        answer = []
        for val, idx in queries:
            if nums[idx] % 2 == 0:                      # gỡ đóng góp của giá trị cũ
                s -= nums[idx]
            nums[idx] += val
            if nums[idx] % 2 == 0:                      # thêm đóng góp của giá trị mới
                s += nums[idx]
            answer.append(s)
        return answer

sol_incremental = Solution().sumEvenAfterQueries
for nums, queries, expected in examples:
    assert sol_incremental(list(nums), queries) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n + q)`, bộ nhớ `O(1)` thêm (ngoài đáp án).

---

### 7.2. Cách 3 — Phân tích bốn trường hợp chẵn/lẻ

**Ý tưởng khác.** Chỉ có bốn trường hợp theo tính chẵn lẻ của `old = nums[idx]` và `val`:

| `old` | `val` | `new = old + val` | Thay đổi của `s` |
|---|---|---|---|
| chẵn | chẵn | chẵn | `+ val` (đóng góp từ `old` thành `old + val`) |
| chẵn | lẻ | lẻ | `- old` (mất đóng góp) |
| lẻ | chẵn | lẻ | `0` |
| lẻ | lẻ | chẵn | `+ (old + val)` (có thêm đóng góp mới) |

**Giải thích từng đoạn.** Cách này làm đúng việc của Cách 2 nhưng gộp "trừ cũ, cộng mới" thành một phép cộng duy nhất tùy trường hợp.

In [9]:
def sum_even_cases(nums: List[int], queries: List[List[int]]) -> List[int]:
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    answer = []
    for val, idx in queries:
        old = nums[idx]
        if old % 2 == 0:
            if val % 2 == 0:
                s += val                       # chẵn + chẵn = chẵn
            else:
                s -= old                       # chẵn + lẻ = lẻ
        else:
            if val % 2 != 0:
                s += old + val                 # lẻ + lẻ = chẵn
            # lẻ + chẵn = lẻ: không đổi
        nums[idx] = old + val
        answer.append(s)
    return answer

for nums, queries, expected in examples:
    got = sum_even_cases(nums, queries)
    print(f"nums={nums}: 4 trường hợp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

nums=[1, 2, 3, 4]: 4 trường hợp = [8, 6, 2, 4] OK
nums=[1]: 4 trường hợp = [0] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def sumEvenAfterQueries(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        s = sum(x for x in nums if x % 2 == 0)
        answer = []
        for val, idx in queries:
            old = nums[idx]
            if old % 2 == 0:
                if val % 2 == 0:
                    s += val                    # chẵn + chẵn = chẵn
                else:
                    s -= old                    # chẵn + lẻ = lẻ
            elif val % 2 != 0:                  # old lẻ: chỉ khi val cũng lẻ thì tổng mới chẵn
                s += old + val                  # lẻ + lẻ = chẵn
            nums[idx] = old + val
            answer.append(s)
        return answer

sol_cases = Solution().sumEvenAfterQueries
for nums, queries, expected in examples:
    assert sol_cases(list(nums), queries) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(n + q)`, bộ nhớ `O(1)` thêm.

---

### 7.3. Cách 4 — Ngoại tuyến: mảng hiệu theo từng chỉ số

**Ý tưởng khác.** Đóng góp của chỉ số `i` vào tổng chẵn là một **hằng số trên từng đoạn thời gian** giữa hai lần chỉ số đó bị cập nhật. Nếu biết trước mọi truy vấn, với mỗi đoạn thời gian `[l, r]` mà `nums[i]` giữ nguyên giá trị chẵn `c` ta cộng `c` vào `diff[l]` và trừ `c` ở `diff[r + 1]`; cuối cùng **tổng tiền tố** của `diff` cho `s` tại mọi thời điểm. Thời điểm `t = 0` là trước mọi truy vấn, `t = k` là sau `k` truy vấn đầu.

**Giải thích từng đoạn.**

- `cur[i]`: giá trị hiện tại của chỉ số `i` khi duyệt các truy vấn; `since[i]`: thời điểm bắt đầu giữ giá trị đó (ban đầu `0`).
- Truy vấn thứ `t` (đánh số từ `1`) tác động chỉ số `idx`: đóng đoạn `[since[idx], t - 1]` với đóng góp `c = cur[idx]` nếu chẵn (`diff[since] += c`, `diff[t] -= c`), sau đó `cur[idx] += val`, `since[idx] = t`.
- Cuối cùng đóng đoạn còn mở của mọi chỉ số đến thời điểm `q` (`diff[q + 1] -= c`).
- Đáp án là tổng tiền tố tại `t = 1..q`.

In [11]:
def sum_even_offline(nums: List[int], queries: List[List[int]]) -> List[int]:
    n, q = len(nums), len(queries)
    cur = list(nums)
    since = [0] * n
    diff = [0] * (q + 2)
    for t, (val, idx) in enumerate(queries, 1):
        c = cur[idx] if cur[idx] % 2 == 0 else 0
        if c:
            diff[since[idx]] += c                # đoạn thời gian [since, t - 1]
            diff[t] -= c
        cur[idx] += val
        since[idx] = t
    for i in range(n):                           # đóng các đoạn còn mở đến thời điểm q
        c = cur[i] if cur[i] % 2 == 0 else 0
        if c:
            diff[since[i]] += c
            diff[q + 1] -= c
    answer, run = [], 0
    for t in range(q + 1):                       # tổng tiền tố: t = 0 là trạng thái ban đầu (bỏ)
        run += diff[t]
        if t >= 1:
            answer.append(run)
    return answer

for nums, queries, expected in examples:
    got = sum_even_offline(nums, queries)
    print(f"nums={nums}: mảng hiệu = {got}", "OK" if got == expected else "SAI")
    assert got == expected

nums=[1, 2, 3, 4]: mảng hiệu = [8, 6, 2, 4] OK
nums=[1]: mảng hiệu = [0] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def sumEvenAfterQueries(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        n, q = len(nums), len(queries)
        cur = list(nums)                         # giá trị hiện tại của từng chỉ số
        since = [0] * n                          # thời điểm chỉ số bắt đầu giữ giá trị hiện tại
        diff = [0] * (q + 2)                     # mảng hiệu theo thời gian
        for t, (val, idx) in enumerate(queries, 1):
            c = cur[idx] if cur[idx] % 2 == 0 else 0
            if c:
                diff[since[idx]] += c            # đóng đoạn [since, t - 1]
                diff[t] -= c
            cur[idx] += val
            since[idx] = t
        for i in range(n):                       # đóng các đoạn còn mở đến thời điểm q
            c = cur[i] if cur[i] % 2 == 0 else 0
            if c:
                diff[since[i]] += c
                diff[q + 1] -= c
        answer, run = [], 0
        for t in range(q + 1):
            run += diff[t]
            if t >= 1:
                answer.append(run)
        return answer

sol_offline = Solution().sumEvenAfterQueries
for nums, queries, expected in examples:
    assert sol_offline(list(nums), queries) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n + q)`, bộ nhớ `O(n + q)`; hơn thua Cách 2 chỉ ở chỗ minh họa ý tưởng ngoại tuyến (hữu ích khi mọi truy vấn có sẵn và ta muốn dùng cấu trúc theo thời gian).

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 1

In giá trị cũ/mới, đóng góp bị gỡ/thêm và `s` sau mỗi truy vấn.

In [13]:
def trace_incremental(nums, queries):
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    print(f"ban đầu nums={nums}, s={s}")
    answer = []
    for step, (val, idx) in enumerate(queries, 1):
        old = nums[idx]
        removed = old if old % 2 == 0 else 0
        s -= removed
        nums[idx] = old + val
        added = nums[idx] if nums[idx] % 2 == 0 else 0
        s += added
        answer.append(s)
        print(f"truy vấn {step}: nums[{idx}] {old} -> {nums[idx]}; gỡ {removed}, thêm {added} -> s = {s}; nums={nums}")
    return answer

assert trace_incremental([1, 2, 3, 4], [[1, 0], [-3, 1], [-4, 0], [2, 3]]) == [8, 6, 2, 4]

ban đầu nums=[1, 2, 3, 4], s=6
truy vấn 1: nums[0] 1 -> 2; gỡ 0, thêm 2 -> s = 8; nums=[2, 2, 3, 4]
truy vấn 2: nums[1] 2 -> -1; gỡ 2, thêm 0 -> s = 6; nums=[2, -1, 3, 4]
truy vấn 3: nums[0] 2 -> -2; gỡ 2, thêm -2 -> s = 2; nums=[-2, -1, 3, 4]
truy vấn 4: nums[3] 4 -> 6; gỡ 4, thêm 6 -> s = 4; nums=[-2, -1, 3, 6]


### 8.2. Cách 4 cho ví dụ 1: mảng hiệu theo thời gian

In `diff` sau khi xử lý mọi truy vấn, rồi tổng tiền tố.

In [14]:
def trace_offline(nums, queries):
    n, q = len(nums), len(queries)
    cur, since, diff = list(nums), [0] * n, [0] * (q + 2)
    for t, (val, idx) in enumerate(queries, 1):
        c = cur[idx] if cur[idx] % 2 == 0 else 0
        if c:
            diff[since[idx]] += c
            diff[t] -= c
            print(f"truy vấn {t}: chỉ số {idx} giữ giá trị chẵn {c} trong thời gian [{since[idx]}, {t - 1}] -> diff[{since[idx]}] += {c}, diff[{t}] -= {c}")
        else:
            print(f"truy vấn {t}: chỉ số {idx} đang giữ giá trị lẻ {cur[idx]} -> không đóng góp")
        cur[idx] += val
        since[idx] = t
    for i in range(n):
        c = cur[i] if cur[i] % 2 == 0 else 0
        if c:
            diff[since[i]] += c
            diff[q + 1] -= c
            print(f"đóng đoạn cuối: chỉ số {i} giữ {c} từ thời điểm {since[i]} đến {q}")
    print("diff =", diff)
    run, out = 0, []
    for t in range(q + 1):
        run += diff[t]
        out.append(run)
    print("tổng tiền tố (t = 0..q) =", out)
    return out[1:]

assert trace_offline([1, 2, 3, 4], [[1, 0], [-3, 1], [-4, 0], [2, 3]]) == [8, 6, 2, 4]

truy vấn 1: chỉ số 0 đang giữ giá trị lẻ 1 -> không đóng góp
truy vấn 2: chỉ số 1 giữ giá trị chẵn 2 trong thời gian [0, 1] -> diff[0] += 2, diff[2] -= 2
truy vấn 3: chỉ số 0 giữ giá trị chẵn 2 trong thời gian [1, 2] -> diff[1] += 2, diff[3] -= 2
truy vấn 4: chỉ số 3 giữ giá trị chẵn 4 trong thời gian [0, 3] -> diff[0] += 4, diff[4] -= 4
đóng đoạn cuối: chỉ số 0 giữ -2 từ thời điểm 3 đến 4
đóng đoạn cuối: chỉ số 3 giữ 6 từ thời điểm 4 đến 4
diff = [6, 2, -2, -4, 2, -4]
tổng tiền tố (t = 0..q) = [6, 8, 6, 2, 4]


### 8.3. Khi `val` là số lẻ và giá trị cũ là số lẻ

Ví dụ `nums = [3]`, `queries = [[5, 0], [3, 0], [-8, 0]]`: `3 -> 8` (lẻ + lẻ = chẵn, `s = 8`), `8 -> 11` (chẵn + lẻ = lẻ, `s = 0`), `11 -> 3` (lẻ + chẵn = lẻ, `s = 0`).

In [15]:
got = Solution().sumEvenAfterQueries([3], [[5, 0], [3, 0], [-8, 0]])
print(got)
assert got == [8, 0, 0] == sum_even_brute([3], [[5, 0], [3, 0], [-8, 0]])

[8, 0, 0]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (cập nhật tăng dần: trừ cũ, cộng mới): ngắn, `O(n + q)`.

```python
class Solution:
    def sumEvenAfterQueries(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        s = sum(x for x in nums if x % 2 == 0)          # tổng chẵn ban đầu
        answer = []
        for val, idx in queries:
            if nums[idx] % 2 == 0:                      # gỡ đóng góp của giá trị cũ
                s -= nums[idx]
            nums[idx] += val
            if nums[idx] % 2 == 0:                      # thêm đóng góp của giá trị mới
                s += nums[idx]
            answer.append(s)
        return answer
```

In [16]:
class Solution:
    def sumEvenAfterQueries(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        s = sum(x for x in nums if x % 2 == 0)          # tổng chẵn ban đầu
        answer = []
        for val, idx in queries:
            if nums[idx] % 2 == 0:                      # gỡ đóng góp của giá trị cũ
                s -= nums[idx]
            nums[idx] += val
            if nums[idx] % 2 == 0:                      # thêm đóng góp của giá trị mới
                s += nums[idx]
            answer.append(s)
        return answer

sol = Solution()
for nums, queries, expected in examples:
    got = sol.sumEvenAfterQueries(list(nums), queries)
    print(f"nums={nums}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

nums=[1, 2, 3, 4]: [8, 6, 2, 4] OK
nums=[1]: [0] OK


## 10. Vì sao đúng

**Bất biến.** Ngay trước mỗi truy vấn, biến `s` bằng tổng các phần tử chẵn của `nums` hiện tại.

- **Khởi tạo:** `s` được tính từ mảng ban đầu theo đúng định nghĩa.
- **Duy trì:** mỗi truy vấn chỉ đổi **đúng một** phần tử `nums[idx]` từ `old` thành `new = old + val`; mọi phần tử khác giữ nguyên nên đóng góp của chúng không đổi. Đóng góp của phần tử đó vào tổng chẵn đổi từ `c(old)` thành `c(new)` với `c(x) = x` nếu `x` chẵn, ngược lại `0`. Cách 2 trừ `c(old)` rồi cộng `c(new)` nên `s_mới = s_cũ - c(old) + c(new)`: đúng bằng tổng chẵn của mảng mới.
- **Kết thúc:** `answer[i]` là `s` sau truy vấn `i`, bằng tổng chẵn sau `i` lần cập nhật: đúng yêu cầu.

**Cách 3** chỉ khai triển `c(new) - c(old)` theo 4 trường hợp tính chẵn lẻ (chẵn+chẵn=chẵn, chẵn+lẻ=lẻ, lẻ+chẵn=lẻ, lẻ+lẻ=chẵn), nên bằng Cách 2.

**Cách 4** viết đóng góp `c(cur[i])` của chỉ số `i` như một hằng số trên mỗi khoảng thời gian chỉ số đó không bị đổi; tổng tiền tố của mảng hiệu tại thời điểm `t` cộng đúng các hằng số của các khoảng chứa `t`, tức là tổng `c(giá trị tại thời điểm t)` trên mọi chỉ số: chính là tổng chẵn sau `t` truy vấn.

**Số âm.** Python cho `x % 2 == 0` đúng với mọi số nguyên; đóng góp của số chẵn âm là số âm, nên `s` có thể âm.

**Kiểm chứng bằng máy.** Oracle độc lập (Cách 1) tính lại tổng chẵn từ mảng **mỗi** truy vấn. Ta đối chiếu mọi cách trên rất nhiều ca ngẫu nhiên có seed (cỡ nhỏ với nhiều số âm, và cỡ đủ lớn), cộng thêm kiểm tra tính chất.

In [17]:
rng = random.Random(1985)
for _ in range(300):
    n = rng.randint(1, 8)
    nums = [rng.randint(-10, 10) for _ in range(n)]
    queries = [[rng.randint(-10, 10), rng.randrange(n)] for _ in range(rng.randint(1, 12))]
    ref = sum_even_brute(nums, queries)
    assert sum_even_incremental(nums, queries) == sum_even_cases(nums, queries) == sum_even_offline(nums, queries) == ref
    assert sol.sumEvenAfterQueries(list(nums), queries) == ref
print("300 ca ngẫu nhiên nhỏ: các cách khớp oracle")

300 ca ngẫu nhiên nhỏ: các cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space (thêm) |
|---|---|---|
| 1. Quét lại cả mảng | `O(n * q)` | `O(1)` |
| 2. Trừ cũ, cộng mới | `O(n + q)` | `O(1)` |
| 3. Bốn trường hợp | `O(n + q)` | `O(1)` |
| 4. Ngoại tuyến + mảng hiệu | `O(n + q)` | `O(n + q)` |

Đo ở cỡ tối đa (`n = q = 10^4`), với giá trị ngẫu nhiên và với trường hợp mọi truy vấn dồn vào một chỉ số:

In [18]:
rng = random.Random(11)
n = q = 10 ** 4
nums = [rng.randint(-10 ** 4, 10 ** 4) for _ in range(n)]
queries = [[rng.randint(-10 ** 4, 10 ** 4), rng.randrange(n)] for _ in range(q)]
one_index = [[rng.randint(-10 ** 4, 10 ** 4), 0] for _ in range(q)]
for label, qs in (("ngẫu nhiên", queries), ("dồn một chỉ số", one_index)):
    ref = sum_even_incremental(nums, qs)
    for name, fn in (("Cách 2", sol_incremental), ("Cách 3", sol_cases), ("Cách 4", sol_offline), ("nộp", sol.sumEvenAfterQueries)):
        t0 = time.perf_counter()
        got = fn(list(nums), qs)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, name
        print(f"{label:16} {name:7}: {dt:7.2f} ms")
t0 = time.perf_counter()
ref_brute = sum_even_brute(nums[:3000], [[v, i % 3000] for v, i in queries[:3000]])
print(f"Cách 1 trên n = q = 3000: {(time.perf_counter() - t0) * 1000:8.1f} ms (để so sánh)")

ngẫu nhiên       Cách 2 :    2.54 ms
ngẫu nhiên       Cách 3 :    2.39 ms
ngẫu nhiên       Cách 4 :    4.97 ms
ngẫu nhiên       nộp    :    2.51 ms
dồn một chỉ số   Cách 2 :    1.89 ms
dồn một chỉ số   Cách 3 :    1.78 ms
dồn một chỉ số   Cách 4 :    3.17 ms
dồn một chỉ số   nộp    :    1.84 ms


Cách 1 trên n = q = 3000:    457.3 ms (để so sánh)


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [19]:
def all_solutions():
    return {"incremental": sum_even_incremental, "cases": sum_even_cases, "offline": sum_even_offline,
            "sol_incremental": lambda a, b: sol_incremental(list(a), b), "sol_cases": lambda a, b: sol_cases(list(a), b),
            "sol_offline": lambda a, b: sol_offline(list(a), b), "final": lambda a, b: sol.sumEvenAfterQueries(list(a), b)}

def check_all(nums, queries, expected=None):
    nums0 = list(nums)
    ref = sum_even_brute(nums, queries)
    for name, fn in all_solutions().items():
        assert fn(list(nums), [list(x) for x in queries]) == ref, (name, nums, queries)
    assert nums == nums0
    if expected is not None:
        assert ref == expected, (nums, queries, ref, expected)
    return ref

# 12.1 ví dụ đề
for nums, queries, expected in examples:
    check_all(nums, queries, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], [[4, 0]], [0]),                                   # lẻ + chẵn = lẻ: tổng chẵn 0
    ([2], [[0, 0]], [2]),                                   # val = 0: không đổi
    ([2], [[1, 0]], [0]),                                   # chẵn -> lẻ: mất đóng góp
    ([1], [[1, 0]], [2]),                                   # lẻ -> chẵn: có thêm đóng góp
    ([0], [[0, 0], [2, 0], [-2, 0]], [0, 2, 0]),            # số 0 là chẵn (đóng góp 0)
    ([1, 3, 5], [[2, 0], [2, 1], [2, 2]], [0, 0, 0]),       # toàn lẻ, cộng chẵn: vẫn lẻ
    ([1, 3, 5], [[1, 0], [1, 1], [1, 2]], [2, 6, 12]),      # toàn lẻ, cộng lẻ: lần lượt thành chẵn 2, 4, 6
    ([2, 4, 6], [[1, 0], [1, 1], [1, 2]], [10, 6, 0]),      # toàn chẵn, cộng lẻ: lần lượt mất 2, 4, 6
    ([-2, -4], [[0, 0], [-2, 1]], [-6, -8]),                # chẵn âm: tổng âm
    ([-3], [[-1, 0], [1, 0], [-3, 0]], [-4, 0, -6]),        # lẻ âm: -3 -> -4 (chẵn, -4), -4 -> -3 (lẻ, 0), -3 -> -6 (chẵn, -6)
    ([10 ** 4, 10 ** 4], [[10 ** 4, 0], [-10 ** 4, 1]], [3 * 10 ** 4, 2 * 10 ** 4]),  # biên giá trị
    ([1, 2, 3], [[1, 1], [1, 1], [1, 1]], [0, 4, 0]),      # cùng chỉ số nhiều lần: 2->3 (lẻ), 3->4 (chẵn), 4->5 (lẻ)
]
for nums, queries, expected in edge:
    got = sum_even_brute(nums, queries)
    if got != expected:
        print("kỳ vọng viết tay chưa khớp oracle:", nums, queries, expected, got)
    check_all(nums, queries)
print("12.2 edge case: OK", len(edge), "ca (mọi cách khớp oracle)")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca (mọi cách khớp oracle)


In [20]:
# 12.3 kiểm tra riêng kỳ vọng viết tay của các ca số âm (đối chiếu tay, không chỉ dựa vào oracle)
assert sum_even_brute([-3], [[-1, 0], [1, 0], [-3, 0]]) == [-4, 0, -6]      # -3 -> -4 (chẵn, -4) ; -4 -> -3 (lẻ, 0) ; -3 -> -6 (chẵn, -6)
assert sum_even_brute([1, 3, 5], [[1, 0], [1, 1], [1, 2]]) == [2, 6, 12]
assert sum_even_brute([2, 4, 6], [[1, 0], [1, 1], [1, 2]]) == [10, 6, 0]
assert sum_even_brute([1, 2, 3], [[1, 1], [1, 1], [1, 1]]) == [0, 4, 0]
assert sum_even_brute([10 ** 4, 10 ** 4], [[10 ** 4, 0], [-10 ** 4, 1]]) == [3 * 10 ** 4, 2 * 10 ** 4]
print("12.3 kỳ vọng tay cho ca số âm / biên: OK")

12.3 kỳ vọng tay cho ca số âm / biên: OK


In [21]:
# 12.4 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed so với oracle)
def wrong_no_subtract(nums, queries):               # quên trừ giá trị cũ
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    out = []
    for val, idx in queries:
        nums[idx] += val
        if nums[idx] % 2 == 0:
            s += nums[idx]
        out.append(s)
    return out

def wrong_parity_of_val(nums, queries):             # chỉ xét tính chẵn của val, không phải giá trị mới
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    out = []
    for val, idx in queries:
        if nums[idx] % 2 == 0:
            s -= nums[idx]
        nums[idx] += val
        if val % 2 == 0:
            s += nums[idx]
        out.append(s)
    return out

def wrong_no_update(nums, queries):                 # quên cập nhật nums[idx]
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    out = []
    for val, idx in queries:
        if nums[idx] % 2 == 0:
            s -= nums[idx]
        if (nums[idx] + val) % 2 == 0:
            s += nums[idx] + val
        out.append(s)
    return out

def wrong_initial_zero(nums, queries):              # quên tính tổng chẵn ban đầu
    nums = list(nums)
    s = 0
    out = []
    for val, idx in queries:
        if nums[idx] % 2 == 0:
            s -= nums[idx]
        nums[idx] += val
        if nums[idx] % 2 == 0:
            s += nums[idx]
        out.append(s)
    return out

def wrong_no_add_new(nums, queries):                # quên cộng giá trị mới (chỉ gỡ giá trị cũ)
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    out = []
    for val, idx in queries:
        if nums[idx] % 2 == 0:
            s -= nums[idx]
        nums[idx] += val
        out.append(s)
    return out

def c_rem(x, m=2):                                  # số dư kiểu C/Java: dấu theo số bị chia
    return int(math.fmod(x, m))

def wrong_c_style_odd(nums, queries):               # cách 3 nhưng kiểm tra "lẻ" bằng số dư kiểu C == 1 (sai với số âm)
    nums = list(nums)
    s = sum(x for x in nums if c_rem(x) == 0)
    out = []
    for val, idx in queries:
        old = nums[idx]
        is_odd_old = c_rem(old) == 1                # số lẻ âm cho c_rem == -1: bị coi là chẵn
        is_odd_val = c_rem(val) == 1
        if not is_odd_old:
            if not is_odd_val:
                s += val
            else:
                s -= old
        elif is_odd_val:
            s += old + val
        nums[idx] = old + val
        out.append(s)
    return out

def find_counter(wrong, seed=9, tries=30000):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 4)
        nums = [rng.randint(-5, 5) for _ in range(n)]
        queries = [[rng.randint(-5, 5), rng.randrange(n)] for _ in range(rng.randint(1, 4))]
        if wrong(list(nums), [list(x) for x in queries]) != sum_even_brute(nums, queries):
            return nums, queries
    return None

counters = {}
for name, fn in (("quên trừ giá trị cũ", wrong_no_subtract), ("chỉ xét chẵn của val", wrong_parity_of_val),
                 ("quên cập nhật nums[idx]", wrong_no_update), ("quên tổng chẵn ban đầu", wrong_initial_zero),
                 ("quên cộng giá trị mới", wrong_no_add_new), ("kiểm tra lẻ kiểu C (số âm)", wrong_c_style_odd)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    nums, queries = case
    print(f"{name:30} nums={nums} queries={queries}: sai -> {fn(list(nums), queries)}, đúng -> {sum_even_brute(nums, queries)}")
    check_all(nums, queries)
print("12.4 phản ví dụ: OK")

quên trừ giá trị cũ            nums=[4, 0, -1, -3] queries=[[5, 0], [0, 3]]: sai -> [4, 4], đúng -> [0, 0]
chỉ xét chẵn của val           nums=[4, 0, -1, -3] queries=[[5, 0], [0, 3]]: sai -> [0, -3], đúng -> [0, 0]
quên cập nhật nums[idx]        nums=[1, -4] queries=[[-2, 1], [-1, 1], [-5, 0]]: sai -> [-6, -2, -6], đúng -> [-6, 0, -4]
quên tổng chẵn ban đầu         nums=[4, 0, -1, -3] queries=[[5, 0], [0, 3]]: sai -> [-4, -4], đúng -> [0, 0]
quên cộng giá trị mới          nums=[1, -4] queries=[[-2, 1], [-1, 1], [-5, 0]]: sai -> [0, 6, 6], đúng -> [-6, 0, -4]
kiểm tra lẻ kiểu C (số âm)     nums=[1, -3, -3, -2] queries=[[-4, 1]]: sai -> [-6], đúng -> [-2]
12.4 phản ví dụ: OK


In [22]:
# 12.5 ngẫu nhiên: nhiều dạng (nhiều số âm, miền hẹp để va chạm, miền đủ lớn theo ràng buộc)
rng = random.Random(4985)
for _ in range(600):
    lim = rng.choice([2, 5, 20, 10 ** 4])
    n = rng.randint(1, 15)
    nums = [rng.randint(-lim, lim) for _ in range(n)]
    queries = [[rng.randint(-lim, lim), rng.randrange(n)] for _ in range(rng.randint(1, 25))]
    check_all(nums, queries)
print("12.5 ngẫu nhiên (600 ca): OK")

# 12.6 cỡ lớn: n = q = 10^4, so các cách nhanh với nhau, oracle trên mẫu nhỏ hơn
rng = random.Random(5985)
for lim in (10 ** 4, 3):
    n = q = 10 ** 4
    nums = [rng.randint(-lim, lim) for _ in range(n)]
    queries = [[rng.randint(-lim, lim), rng.randrange(n)] for _ in range(q)]
    outs = [fn(nums, queries) for name, fn in all_solutions().items()]
    assert all(o == outs[0] for o in outs)
    sub_nums, sub_q = nums[:200], [[v, i % 200] for v, i in queries[:300]]
    check_all(sub_nums, sub_q)
print("12.6 cỡ lớn (n = q = 10^4): OK")

# 12.7 tính chất: mọi đáp án chẵn; val = 0 không đổi; thêm một truy vấn cuối không đổi các đáp án trước; tổng chẵn khớp trạng thái cuối
rng = random.Random(7985)
for _ in range(300):
    n = rng.randint(1, 12)
    nums = [rng.randint(-20, 20) for _ in range(n)]
    queries = [[rng.randint(-20, 20), rng.randrange(n)] for _ in range(rng.randint(1, 20))]
    ans = sol.sumEvenAfterQueries(list(nums), queries)
    assert all(a % 2 == 0 for a in ans)
    assert sol.sumEvenAfterQueries(list(nums), queries + [[rng.randint(-20, 20), rng.randrange(n)]])[:-1] == ans
    zero_q = queries + [[0, rng.randrange(n)]]
    z = sol.sumEvenAfterQueries(list(nums), zero_q)
    assert z[-1] == z[-2]
    final = list(nums)
    for v, i in queries:
        final[i] += v
    assert ans[-1] == sum(x for x in final if x % 2 == 0)
print("12.7 tính chất: OK")

12.5 ngẫu nhiên (600 ca): OK


12.6 cỡ lớn (n = q = 10^4): OK


12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ giá trị cũ

Chỉ cộng giá trị mới (nếu chẵn) mà không gỡ giá trị cũ (nếu chẵn) làm `s` đếm hai lần cùng một ô.

In [23]:
nums, queries = counters["quên trừ giá trị cũ"]
print("nums =", nums, "queries =", queries, "| sai ->", wrong_no_subtract(nums, queries), "| đúng ->", sum_even_brute(nums, queries))
assert wrong_no_subtract(nums, queries) != Solution().sumEvenAfterQueries(list(nums), queries)

nums = [4, 0, -1, -3] queries = [[5, 0], [0, 3]] | sai -> [4, 4] | đúng -> [0, 0]


Vì sao sai: giá trị cũ chẵn vẫn nằm trong `s` sau khi bị thay. **Đúng:** `if nums[idx] % 2 == 0: s -= nums[idx]` **trước** khi cập nhật.

### Lỗi 2 — Chỉ xét tính chẵn của `val`

Tổng `old + val` chẵn khi `old` và `val` **cùng tính chẵn lẻ**, không chỉ khi `val` chẵn.

In [24]:
nums, queries = counters["chỉ xét chẵn của val"]
print("nums =", nums, "queries =", queries, "| sai ->", wrong_parity_of_val(nums, queries), "| đúng ->", sum_even_brute(nums, queries))
assert wrong_parity_of_val(nums, queries) != Solution().sumEvenAfterQueries(list(nums), queries)

nums = [4, 0, -1, -3] queries = [[5, 0], [0, 3]] | sai -> [0, -3] | đúng -> [0, 0]


Vì sao sai: lẻ + lẻ cho chẵn, chẵn + lẻ cho lẻ. **Đúng:** kiểm tra `nums[idx] % 2 == 0` trên **giá trị mới** sau khi cập nhật.

### Lỗi 3 — Quên cập nhật `nums[idx]`

Nếu chỉ tính `old + val` mà không ghi lại vào mảng, các truy vấn sau dùng giá trị cũ.

In [25]:
nums, queries = counters["quên cập nhật nums[idx]"]
print("nums =", nums, "queries =", queries, "| sai ->", wrong_no_update(nums, queries), "| đúng ->", sum_even_brute(nums, queries))
assert wrong_no_update(nums, queries) != Solution().sumEvenAfterQueries(list(nums), queries)

nums = [1, -4] queries = [[-2, 1], [-1, 1], [-5, 0]] | sai -> [-6, -2, -6] | đúng -> [-6, 0, -4]


Vì sao sai: mỗi truy vấn cộng vào **giá trị hiện tại** (đã gồm các truy vấn trước). **Đúng:** `nums[idx] += val`.

### Lỗi 4 — Quên tính tổng chẵn ban đầu

Khởi tạo `s = 0` làm mất đóng góp của các số chẵn ban đầu chưa bị truy vấn nào chạm tới.

In [26]:
nums, queries = counters["quên tổng chẵn ban đầu"]
print("nums =", nums, "queries =", queries, "| sai ->", wrong_initial_zero(nums, queries), "| đúng ->", sum_even_brute(nums, queries))
assert wrong_initial_zero(nums, queries) != Solution().sumEvenAfterQueries(list(nums), queries)

nums = [4, 0, -1, -3] queries = [[5, 0], [0, 3]] | sai -> [-4, -4] | đúng -> [0, 0]


Vì sao sai: ví dụ 1 có `2` và `4` chẵn ngay từ đầu. **Đúng:** `s = sum(x for x in nums if x % 2 == 0)` trước vòng lặp.

### Lỗi 5 — Quên cộng giá trị mới

Chỉ gỡ giá trị cũ mà quên cộng giá trị mới (khi nó chẵn).

In [27]:
nums, queries = counters["quên cộng giá trị mới"]
print("nums =", nums, "queries =", queries, "| sai ->", wrong_no_add_new(nums, queries), "| đúng ->", sum_even_brute(nums, queries))
assert wrong_no_add_new(nums, queries) != Solution().sumEvenAfterQueries(list(nums), queries)

nums = [1, -4] queries = [[-2, 1], [-1, 1], [-5, 0]] | sai -> [0, 6, 6] | đúng -> [-6, 0, -4]


Vì sao sai: giá trị mới chẵn phải được cộng vào `s`. **Đúng:** gỡ cũ **và** thêm mới.

### Lỗi 6 — Kiểm tra lẻ bằng `x % 2 == 1` với phép chia dư kiểu C/Java

Trong C/C++/Java `-3 % 2 == -1`, nên `x % 2 == 1` bỏ sót số lẻ âm. Python không mắc lỗi này (`-3 % 2 == 1`), nhưng nếu bạn chuyển lời giải sang ngôn ngữ khác thì phải dùng `x % 2 != 0` hoặc `(x & 1) == 1`. Ta mô phỏng số dư kiểu C bằng `math.fmod`.

In [28]:
nums, queries = counters["kiểm tra lẻ kiểu C (số âm)"]
print("nums =", nums, "queries =", queries, "| sai (kiểu C) ->", wrong_c_style_odd(nums, queries), "| đúng ->", sum_even_brute(nums, queries))
assert wrong_c_style_odd(nums, queries) != Solution().sumEvenAfterQueries(list(nums), queries)
print("Python an toàn:", -3 % 2 == 1, "| kiểu C:", c_rem(-3), "-> cần dùng 'x % 2 != 0' để kiểm tra lẻ")

nums = [1, -3, -3, -2] queries = [[-4, 1]] | sai (kiểu C) -> [-6] | đúng -> [-2]
Python an toàn: True | kiểu C: -1 -> cần dùng 'x % 2 != 0' để kiểm tra lẻ


Vì sao sai: số lẻ âm bị coi như không lẻ. **Đúng:** kiểm tra chẵn bằng `x % 2 == 0` (đúng ở mọi ngôn ngữ), và lẻ là phủ định của nó.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "cập nhật một phần tử rồi báo thống kê của toàn mảng" ⇒ **cập nhật tăng dần**: thống kê đổi đúng bằng đóng góp mới trừ đóng góp cũ của phần tử đó.
- Khuôn mẫu: `gỡ cũ → cập nhật → thêm mới`; luôn gỡ **trước** khi cập nhật.
- Kiểm tra chẵn bằng `x % 2 == 0` (đúng cho số âm, `0` là chẵn); lẻ là `not chẵn`.
- Giữ bất biến "biến `s` luôn đúng với mảng hiện tại" rồi chứng minh bằng quy nạp.
- Khi mọi truy vấn có sẵn, có thể làm **ngoại tuyến** bằng mảng hiệu theo thời gian.
- Cùng họ: Range Sum Query - Mutable (307), Running Sum of 1d Array (1480), Product of Array Except Self (238).
- Test nên có: số `0`, số âm, truy vấn cùng một chỉ số liên tiếp, toàn lẻ, toàn chẵn, `val = 0`, giá trị biên `±10^4`.

## 15. Biến thể

1. **Tổng các số lẻ**: `s_lẻ = tổng_toàn_bộ - s_chẵn`, hoặc cập nhật tương tự với `x % 2 != 0`.
2. **Số lượng số chẵn** sau mỗi truy vấn (bài tập 5).
3. **Tổng các số chia hết cho `k`:** thay `% 2 == 0` bằng `% k == 0`; cùng cách cập nhật tăng dần.
4. Bài liên quan: 307 (truy vấn đoạn có cập nhật, cần cây Fenwick/segment tree), 1480, 238.

Biến thể 3 (chia hết cho `k`), kiểm bằng oracle quét lại:

In [29]:
def sum_divisible_incremental(nums: List[int], queries: List[List[int]], k: int) -> List[int]:
    nums = list(nums)
    s = sum(x for x in nums if x % k == 0)
    out = []
    for val, idx in queries:
        if nums[idx] % k == 0:
            s -= nums[idx]
        nums[idx] += val
        if nums[idx] % k == 0:
            s += nums[idx]
        out.append(s)
    return out

def sum_divisible_brute(nums, queries, k):
    nums = list(nums)
    out = []
    for val, idx in queries:
        nums[idx] += val
        out.append(sum(x for x in nums if x % k == 0))
    return out

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 8)
    k = rng.randint(1, 7)
    nums = [rng.randint(-30, 30) for _ in range(n)]
    queries = [[rng.randint(-30, 30), rng.randrange(n)] for _ in range(rng.randint(1, 10))]
    assert sum_divisible_incremental(nums, queries, k) == sum_divisible_brute(nums, queries, k)
    if k == 2:
        assert sum_divisible_incremental(nums, queries, 2) == sum_even_brute(nums, queries)
print("biến thể chia hết cho k: khớp oracle; k = 2 chính là bài này")

biến thể chia hết cho k: khớp oracle; k = 2 chính là bài này


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét lại | Tính lại tổng chẵn mỗi truy vấn | `O(n * q)` | `O(1)` | Oracle |
| 2. Trừ cũ, cộng mới | Cập nhật tăng dần | `O(n + q)` | `O(1)` | **Cách nộp chính** |
| 3. Bốn trường hợp | Phân tích chẵn/lẻ `(old, val)` | `O(n + q)` | `O(1)` | Hiểu cơ chế |
| 4. Ngoại tuyến | Mảng hiệu theo thời gian | `O(n + q)` | `O(n + q)` | Khi có sẵn mọi truy vấn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(-3 % 2, -4 % 2, (-3) & 1, 0 % 2 == 0)
print(sum(x for x in [1, 2, 3, 4] if x % 2 == 0), sum(x for x in [-2, 5, -6] if x % 2 == 0))
```

In [30]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # bốn giá trị của dòng 1
my_guess_2 = None      # hai số của dòng 2

### Bài 2 — Tính tay (dễ-vừa)

`nums = [2, 3, 5]`, `queries = [[1, 1], [2, 0], [-5, 2]]`. Ghi mảng đáp án (theo từng truy vấn, giữ nguyên thứ tự).

In [31]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cập nhật tăng dần.

In [32]:
def exercise_3(nums: List[int], queries: List[List[int]]) -> List[int]:
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    answer = []
    for val, idx in queries:
        # TODO 1: nếu nums[idx] chẵn thì gỡ khỏi s
        # TODO 2: cập nhật nums[idx]
        # TODO 3: nếu nums[idx] mới chẵn thì cộng vào s
        # TODO 4: ghi s vào answer
        pass
    return answer


# for nums, queries, expected in examples:
#     assert exercise_3(nums, queries) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(nums, queries)` bằng **bốn trường hợp chẵn/lẻ** của `(old, val)`.

In [33]:
def exercise_4(nums: List[int], queries: List[List[int]]) -> List[int]:
    # TODO
    pass


# for nums, queries, expected in examples:
#     assert exercise_4(nums, queries) == expected

### Bài 5 — Mini challenge: đếm số lượng số chẵn (khó)

Viết `exercise_5(nums, queries)` trả về, sau mỗi truy vấn, **số lượng** phần tử chẵn của `nums` (không phải tổng), trong `O(n + q)`.

```python
exercise_5([1, 2, 3, 4], [[1, 0], [-3, 1], [-4, 0], [2, 3]]) -> [3, 2, 2, 2]
exercise_5([1], [[4, 0]])                                     -> [0]
exercise_5([2, 4], [[1, 0], [1, 0]])                          -> [1, 2]
```
Gợi ý: đếm tăng dần, `cnt -= 1` nếu giá trị cũ chẵn, `cnt += 1` nếu giá trị mới chẵn.

In [34]:
def exercise_5(nums: List[int], queries: List[List[int]]) -> List[int]:
    # TODO
    pass


# assert exercise_5([1, 2, 3, 4], [[1, 0], [-3, 1], [-4, 0], [2, 3]]) == [3, 2, 2, 2]
# assert exercise_5([1], [[4, 0]]) == [0]
# assert exercise_5([2, 4], [[1, 0], [1, 0]]) == [1, 2]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: trong Python `%` cho kết quả cùng dấu với số chia (`2`), nên không âm.
- Mức 2: `-3 % 2 = 1`, `-4 % 2 = 0`, `(-3) & 1 = 1`, và `0` là chẵn.
- Mức 3: dòng 1: `1 0 1 True`; dòng 2: `6 -8` (`2 + 4` và `-2 + -6`).

**Bài 2**
- Mức 1: tổng chẵn ban đầu của `[2, 3, 5]` là `2`.
- Mức 2: `[2,4,5]` -> `6`; `[4,4,5]` -> `8`; `[4,4,0]` -> `8`.
- Mức 3: `[6, 8, 8]`.

**Bài 3**
- Mức 1: `if nums[idx] % 2 == 0: s -= nums[idx]`.
- Mức 2: `nums[idx] += val`.
- Mức 3: `if nums[idx] % 2 == 0: s += nums[idx]`; `answer.append(s)`.

**Bài 4**
- Mức 1: `old = nums[idx]`; xét `old % 2 == 0` và `val % 2 == 0`.
- Mức 2: chẵn+chẵn: `s += val`; chẵn+lẻ: `s -= old`; lẻ+lẻ: `s += old + val`; lẻ+chẵn: không đổi.
- Mức 3: nhớ `nums[idx] = old + val` ở cuối.

**Bài 5**
- Mức 1: `cnt = sum(1 for x in nums if x % 2 == 0)`.
- Mức 2: với mỗi truy vấn: `if nums[idx] % 2 == 0: cnt -= 1`, cập nhật, `if nums[idx] % 2 == 0: cnt += 1`.
- Mức 3: `answer.append(cnt)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [35]:
print(-3 % 2, -4 % 2, (-3) & 1, 0 % 2 == 0)
print(sum(x for x in [1, 2, 3, 4] if x % 2 == 0), sum(x for x in [-2, 5, -6] if x % 2 == 0))

assert (-3 % 2, -4 % 2, (-3) & 1, 0 % 2 == 0) == (1, 0, 1, True)
assert (sum(x for x in [1, 2, 3, 4] if x % 2 == 0), sum(x for x in [-2, 5, -6] if x % 2 == 0)) == (6, -8)

1 0 1 True
6 -8


### Đáp án Bài 2

`[2,3,5]`: `s = 2`. Truy vấn 1: `nums[1]` từ `3` (lẻ) thành `4` (chẵn): `s = 2 + 4 = 6`. Truy vấn 2: `nums[0]` từ `2` thành `4`: `s = 6 - 2 + 4 = 8`. Truy vấn 3: `nums[2]` từ `5` (lẻ) thành `0` (chẵn): `s = 8 + 0 = 8`.

In [36]:
hand_answer_solution = [6, 8, 8]
assert Solution().sumEvenAfterQueries([2, 3, 5], [[1, 1], [2, 0], [-5, 2]]) == hand_answer_solution
assert sum_even_brute([2, 3, 5], [[1, 1], [2, 0], [-5, 2]]) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [37]:
def exercise_3_solution(nums: List[int], queries: List[List[int]]) -> List[int]:
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    answer = []
    for val, idx in queries:
        if nums[idx] % 2 == 0:
            s -= nums[idx]
        nums[idx] += val
        if nums[idx] % 2 == 0:
            s += nums[idx]
        answer.append(s)
    return answer

for nums, queries, expected in examples:
    assert exercise_3_solution(nums, queries) == expected
rng = random.Random(3)
for _ in range(200):
    n = rng.randint(1, 8)
    nums = [rng.randint(-10, 10) for _ in range(n)]
    queries = [[rng.randint(-10, 10), rng.randrange(n)] for _ in range(rng.randint(1, 10))]
    assert exercise_3_solution(nums, queries) == sum_even_brute(nums, queries)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [38]:
def exercise_4_solution(nums: List[int], queries: List[List[int]]) -> List[int]:
    nums = list(nums)
    s = sum(x for x in nums if x % 2 == 0)
    answer = []
    for val, idx in queries:
        old = nums[idx]
        if old % 2 == 0:
            if val % 2 == 0:
                s += val
            else:
                s -= old
        elif val % 2 != 0:
            s += old + val
        nums[idx] = old + val
        answer.append(s)
    return answer

for nums, queries, expected in examples:
    assert exercise_4_solution(nums, queries) == expected
rng = random.Random(4)
for _ in range(200):
    n = rng.randint(1, 8)
    nums = [rng.randint(-10, 10) for _ in range(n)]
    queries = [[rng.randint(-10, 10), rng.randrange(n)] for _ in range(rng.randint(1, 10))]
    assert exercise_4_solution(nums, queries) == sum_even_brute(nums, queries)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [39]:
def exercise_5_solution(nums: List[int], queries: List[List[int]]) -> List[int]:
    nums = list(nums)
    cnt = sum(1 for x in nums if x % 2 == 0)
    answer = []
    for val, idx in queries:
        if nums[idx] % 2 == 0:
            cnt -= 1
        nums[idx] += val
        if nums[idx] % 2 == 0:
            cnt += 1
        answer.append(cnt)
    return answer

def count_even_brute(nums, queries):
    nums = list(nums)
    out = []
    for val, idx in queries:
        nums[idx] += val
        out.append(sum(1 for x in nums if x % 2 == 0))
    return out

assert exercise_5_solution([1, 2, 3, 4], [[1, 0], [-3, 1], [-4, 0], [2, 3]]) == [3, 2, 2, 2]
assert exercise_5_solution([1], [[4, 0]]) == [0]
assert exercise_5_solution([2, 4], [[1, 0], [1, 0]]) == [1, 2]
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 8)
    nums = [rng.randint(-10, 10) for _ in range(n)]
    queries = [[rng.randint(-10, 10), rng.randrange(n)] for _ in range(rng.randint(1, 10))]
    assert exercise_5_solution(nums, queries) == count_even_brute(nums, queries)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (cập nhật tăng dần)

```python
class Solution:
    def sumEvenAfterQueries(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        s = sum(x for x in nums if x % 2 == 0)          # bất biến: s = tổng chẵn của nums hiện tại
        answer = []
        for val, idx in queries:
            if nums[idx] % 2 == 0:                      # gỡ đóng góp của giá trị cũ
                s -= nums[idx]
            nums[idx] += val                            # cập nhật
            if nums[idx] % 2 == 0:                      # thêm đóng góp của giá trị mới
                s += nums[idx]
            answer.append(s)
        return answer
```

Công thức nhớ nhanh: `s_mới = s_cũ - c(old) + c(new)` với `c(x) = x` nếu `x` chẵn, `0` nếu lẻ.

### Checklist trước khi submit

- [ ] Đã tính tổng chẵn **ban đầu** trước vòng lặp?
- [ ] Gỡ giá trị cũ (nếu chẵn) **trước** khi cập nhật, và thêm giá trị mới (nếu chẵn) **sau** khi cập nhật?
- [ ] Kiểm tra chẵn trên **giá trị mới** (không phải chỉ `val`)?
- [ ] Đã ghi lại `nums[idx] += val` để các truy vấn sau thấy giá trị mới?
- [ ] Kiểm tra chẵn bằng `x % 2 == 0` (đúng cho số âm và `0`)?
- [ ] Đã thử: số `0`, số âm, toàn lẻ, truy vấn lặp cùng chỉ số, `val = 0`, biên `±10^4`?